# Voice dataset prep: choose a speaker, dial in pitch/bass, then export for RVC training

This dataset is stored as a Hugging Face `datasets`-library repo (parquet-backed),
not individually browsable files -- so it's loaded with `load_dataset(..., streaming=True)`
and filtered by the `speaker_id` field, not by downloading files matching a path pattern.

**Honest caveat**: whether streaming actually avoids pulling most of the ~17GB depends on
whether rows happen to be grouped by speaker or shuffled across shards -- that isn't
confirmed either way. `MAX_EXAMPLES_TO_SCAN` below is a safety cap so a stage doesn't
silently scan the entire dataset without you knowing.

Three stages:
1. Preview one short clip from as many speakers as the scan cap reaches, to pick a voice
2. Tweak pitch/bass and preview before/after on that one clip -- re-run freely to iterate
3. Once happy, collect that speaker's full set of clips and export them, pitch/bass-adjusted,
   ready to become the training notebook's input dataset

**If you plan to try several different speakers**: streaming from Hugging Face on every
attempt re-pays the scan cost each time. Consider running Stage 1 once with a large scan
cap (or the full dataset), saving what you collect as a Kaggle Dataset, and doing Stages 2-3
against that Kaggle-hosted copy instead -- much faster to iterate against Kaggle's own
storage than re-streaming from Hugging Face each time.

In [ ]:
!pip install -q pedalboard soundfile datasets

## Configure

`REPO_ID`: the Hugging Face dataset repo to pull from -- confirm on its dataset card that it
exposes a `speaker_id` field (schema varies slightly between VCTK mirrors).

In [ ]:
import os

REPO_ID = "CSTR-Edinburgh/vctk"  # swap for whichever VCTK mirror you're using

MAX_EXAMPLES_TO_SCAN = 20000  # safety cap for Stage 1 -- raise if you need more speaker coverage

OUTPUT_DIR = "/kaggle/working/adjusted"

## Stage 1: one clip per speaker

Streams through the dataset, keeping the first example seen for each new `speaker_id`,
up to `MAX_EXAMPLES_TO_SCAN` rows. If VCTK's rows are grouped by speaker, this may not
reach every speaker within the cap -- raise `MAX_EXAMPLES_TO_SCAN` if you're missing ones
you expected to see.

**If this errors with `RuntimeError: Dataset scripts are no longer supported`**: that
repo ships a legacy Python loading script, which recent `datasets` versions refuse to
execute (a security change, not specific to VCTK). `load_vctk_stream()` below handles
this automatically by retrying against Hugging Face's auto-converted parquet branch
(`refs/convert/parquet`), which exists for most script-based repos. If it still fails
after that fallback, the repo likely wasn't auto-converted -- check the "Data Studio"
preview tab on the dataset's Hugging Face page (if that table preview works, the parquet
branch exists) or swap `REPO_ID` for a different VCTK mirror.


In [ ]:
from datasets import load_dataset
from IPython.display import Audio, display


def load_vctk_stream():
    """Some dataset repos still ship a legacy loading script instead of
    plain data files -- recent `datasets` versions refuse to execute those
    (arbitrary code from the hub), raising RuntimeError. Hugging Face
    auto-converts most such repos to parquet on a separate git ref, so
    this retries there instead of failing outright."""
    try:
        return load_dataset(REPO_ID, split="train", streaming=True)
    except RuntimeError as e:
        if "Dataset scripts are no longer supported" not in str(e):
            raise
        print("REPO_ID ships a legacy loading script -- retrying against "
              "the auto-converted parquet branch (refs/convert/parquet)...")
        return load_dataset(
            REPO_ID, split="train", streaming=True, revision="refs/convert/parquet"
        )


stream = load_vctk_stream()

speaker_example = {}
for i, example in enumerate(stream):
    if i >= MAX_EXAMPLES_TO_SCAN:
        break
    speaker_id = example["speaker_id"]
    if speaker_id not in speaker_example:
        speaker_example[speaker_id] = example

print(f"found {len(speaker_example)} distinct speakers within the first {i + 1} rows scanned")


In [ ]:
for speaker_id, example in sorted(speaker_example.items()):
    print(speaker_id, "--", example.get("gender", "?"), example.get("accent", "?"))
    display(Audio(example["audio"]["array"], rate=example["audio"]["sampling_rate"]))

## Stage 2: pick a speaker, tweak pitch/bass, preview before/after

Set `CHOSEN_SPEAKER` to whichever speaker ID sounded closest to what you want above.
Then tweak `PITCH_SEMITONES` / `BASS_GAIN_DB` and **re-run this cell** as many times as
you like -- each run plays the original and the adjusted version so you can compare.

In [ ]:
from pedalboard import Pedalboard, PitchShift, LowShelfFilter

CHOSEN_SPEAKER = "p225"  # pick from the speaker IDs printed in Stage 1

PITCH_SEMITONES = -3.0  # positive = higher, negative = lower -- tweak and re-run
BASS_GAIN_DB = 4.0      # positive = boost bass, negative = cut it -- tweak and re-run


def build_board(pitch_semitones, bass_gain_db):
    effects = []
    if pitch_semitones:
        effects.append(PitchShift(semitones=pitch_semitones))
    if bass_gain_db:
        effects.append(LowShelfFilter(cutoff_frequency_hz=200.0, gain_db=bass_gain_db))
    return Pedalboard(effects)


preview_example = speaker_example[CHOSEN_SPEAKER]
audio = preview_example["audio"]["array"]
sample_rate = preview_example["audio"]["sampling_rate"]
board = build_board(PITCH_SEMITONES, BASS_GAIN_DB)
processed = board(audio.reshape(1, -1).astype("float32"), sample_rate)[0]

print(f"{CHOSEN_SPEAKER} -- original:")
display(Audio(audio, rate=sample_rate))
print(f"{CHOSEN_SPEAKER} -- pitch={PITCH_SEMITONES:+.1f} semitones, bass={BASS_GAIN_DB:+.1f}dB:")
display(Audio(processed, rate=sample_rate))

## Stage 3: collect and export the full chosen speaker's clips

Only run this once you're happy with `PITCH_SEMITONES`/`BASS_GAIN_DB` from Stage 2. Streams
through the dataset again, this time keeping every row matching `CHOSEN_SPEAKER` (up to
`MAX_EXAMPLES_TO_SCAN` rows scanned, same caveat as Stage 1 about speaker grouping).

In [ ]:
import soundfile as sf
from pathlib import Path

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

stream = load_vctk_stream()

written = 0
for i, example in enumerate(stream):
    if i >= MAX_EXAMPLES_TO_SCAN:
        break
    if example["speaker_id"] != CHOSEN_SPEAKER:
        continue

    audio = example["audio"]["array"].astype("float32")
    sample_rate = example["audio"]["sampling_rate"]
    processed = board(audio.reshape(1, -1), sample_rate)[0]

    filename = Path(example.get("file") or f"{CHOSEN_SPEAKER}_{example.get('text_id', written)}.wav").name
    sf.write(str(output_dir / filename), processed, sample_rate)
    written += 1
    if written % 25 == 0:
        print(f"{written} clips written so far...")

print(f"Done: {written} file(s) for {CHOSEN_SPEAKER} written to {output_dir} "
      f"(scanned {i + 1} rows)")
if written == 0:
    print("WARNING: found none -- check CHOSEN_SPEAKER matches an ID from Stage 1, "
          "or raise MAX_EXAMPLES_TO_SCAN if this speaker's rows weren't reached yet.")


## Next step

Use Kaggle's "Save Version" then create a **New Dataset** from this notebook's output
(`/kaggle/working/adjusted`) -- then attach that dataset to your Applio training notebook
instead of the raw source dataset.